# GraphCodeBERT XAI Vulnerability Detector

**Workflow:**
1. Load the latest model from local directory
2. Perform vulnerability detection on test input code
3. Visualize important tokens using XAI techniques

---

## Step 1: Import Libraries

In [0]:
import os
import json
import numpy as np
import torch
from pathlib import Path
from typing import Tuple, List, Dict
from transformers import AutoTokenizer, AutoModel
import re
import html as html_module
from IPython.display import display, HTML

print(f"✅ PyTorch device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## Step 2: Configuration

In [0]:
class Config:
    """Configuration parameters"""
    MODEL_NAME = 'mahdin70/GraphCodeBERT-VulnCWE'
    TOKENIZER_NAME = "microsoft/graphcodebert-base"
    LOCAL_MODEL_DIR = "./models/graphcodebertv2_6ep"
    MAX_LENGTH = 512
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    BATCH_SIZE = 16
    TOP_K_TOKENS = 10
    
    # Color threshold for XAI visualization
    CRITICAL_THRESHOLD = 0.66
    MEDIUM_THRESHOLD = 0.33

print(f"📋 Configuration:")
print(f"   Model directory: {Config.LOCAL_MODEL_DIR}")
print(f"   Max token length: {Config.MAX_LENGTH}")
print(f"   Device: {Config.DEVICE}")
print(f"   Top-K tokens: {Config.TOP_K_TOKENS}")

## Step 3: Load Model

In [0]:
def load_model_local(model_path: str = None) -> Tuple:
    """
    Load model and tokenizer from local path
    
    Args:
        model_path: Path to saved model directory
        
    Returns:
        Tuple of (model, tokenizer, device)
    """
    if model_path is None:
        model_path = Config.LOCAL_MODEL_DIR
    
    model_path = Path(model_path)
    
    if not model_path.exists():
        print(f"❌ Model path not found: {model_path.absolute()}")
        return None, None, None
    
    print(f"📂 Loading model from: {model_path.absolute()}")
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(str(model_path))
        model = AutoModel.from_pretrained(str(model_path), trust_remote_code=True)
        model.to(Config.DEVICE)
        model.eval()
        
        print(f"✅ Model loaded successfully!")
        print(f"   Model parameters: {sum(p.numel() for p in model.parameters()):,}")
        print(f"   Device: {Config.DEVICE}")
        
        return model, tokenizer, Config.DEVICE
    
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return None, None, None

# Load model
print("="*80)
print("🚀 Loading GraphCodeBERT Model (Latest Version)")
print("="*80 + "\n")

model, tokenizer, device = load_model_local()

if model is None:
    print("❌ Failed to load model!")
else:
    print("\n✅ Model ready for inference")

## Step 4: Vulnerability Detection Function

In [0]:
def predict_vulnerability(
    code_text: str,
    model: AutoModel,
    tokenizer: AutoTokenizer,
    device: torch.device
) -> Dict:
    """
    Predict vulnerability for given code snippet
    
    Args:
        code_text: Source code to analyze
        model: Pre-trained model
        tokenizer: Tokenizer
        device: Computation device
        
    Returns:
        Dictionary with prediction results
    """
    model.eval()
    
    try:
        # Tokenize and encode
        inputs = tokenizer(
            code_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=Config.MAX_LENGTH
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Forward pass
        with torch.no_grad():
            outputs = model(**inputs)
            
            # Extract logits
            if 'vul_logits' in outputs:
                logits = outputs['vul_logits']
            else:
                logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
            
            # Handle different tensor shape scenarios
            if logits.dim() == 1:
                # If 1D tensor, unsqueeze to make it [1, num_classes]
                logits = logits.unsqueeze(0)
            elif logits.dim() > 2:
                # If more than 2D, reshape to 2D
                logits = logits.view(-1, logits.size(-1))
            
            # Ensure we have the correct shape [batch_size, num_classes]
            if logits.shape[0] != 1:
                logits = logits[0:1]  # Take only the first sample
            
            # Get probabilities
            probs = torch.softmax(logits, dim=1)[0]  # [num_classes]
            pred_class = torch.argmax(probs).item()
            pred_prob = probs[1].item() if len(probs) > 1 else probs[0].item()  # Vulnerability probability
        
        return {
            'code': code_text,
            'prediction': 'VULNERABLE' if pred_class == 1 else 'SAFE',
            'vulnerability_prob': float(pred_prob),
            'confidence': float(max(probs).item()),
            'raw_logits': logits.cpu().numpy()
        }
    
    except Exception as e:
        print(f"❌ Error during prediction: {e}")
        import traceback
        traceback.print_exc()
        return None

print("✅ Vulnerability detection function defined")


## Step 5: XAI - Token Importance Analysis Functions

In [0]:
# ============================================================================
# Step 5: Gradient-based Saliency Maps (가장 빠른 방법)
# ============================================================================

def compute_gradient_saliency(
    code_text: str,
    model,
    tokenizer: AutoTokenizer,
    device: torch.device
) -> Tuple[List[str], np.ndarray]:
    """
    Compute token importance using gradient-based saliency map
    
    Principle: ∂Loss/∂Input → 어느 토큰의 변화가 손실을 가장 크게 변화시키는가?
    """
    model.eval()
    
    # Get base model and embedding layer (handle different model structures)
    if hasattr(model, 'base'):
        base_model = model.base
    elif hasattr(model, 'model'):
        base_model = model.model
    else:
        base_model = model
    
    # Try to get embedding layer using HuggingFace standard method
    try:
        embedding_layer = model.get_input_embeddings()
    except:
        # Fallback: try common attribute names
        if hasattr(base_model, 'embeddings'):
            embedding_layer = base_model.embeddings.word_embeddings if hasattr(base_model.embeddings, 'word_embeddings') else base_model.embeddings
        else:
            raise AttributeError("Cannot find embedding layer in model")
    
    # 1. Tokenize
    inputs = tokenizer(
        code_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=Config.MAX_LENGTH
    ).to(device)
    
    input_ids = inputs['input_ids']
    
    # 2. Get embeddings with gradient tracking
    with torch.enable_grad():
        # Get embeddings and enable gradients on them (not on input_ids which are integers)
        embeddings = embedding_layer(input_ids)
        embeddings_input = embeddings.clone().detach().requires_grad_(True)
        embeddings_input.retain_grad()
        
        # 3. Forward pass through base model
        try:
            outputs = base_model(
                inputs_embeds=embeddings_input,
                attention_mask=inputs['attention_mask'],
                token_type_ids=inputs.get('token_type_ids', None)
            )
        except TypeError:
            # If inputs_embeds not supported, fall back to input_ids
            outputs = base_model(input_ids=input_ids, attention_mask=inputs['attention_mask'])
        
        # 4. Get model outputs for vulnerability prediction
        if hasattr(outputs, 'logits'):
            logits = outputs.logits
        elif 'vul_logits' in outputs:
            logits = outputs['vul_logits']
        else:
            logits = outputs[0]
        
        # Handle different tensor shapes
        if logits.dim() == 1:
            logits = logits.unsqueeze(0)
        elif logits.dim() > 2:
            logits = logits.view(-1, logits.size(-1))
        
        # 5. Get vulnerability probability
        vuln_probs = torch.softmax(logits, dim=-1)[0]
        vuln_prob = vuln_probs[1] if len(vuln_probs) > 1 else vuln_probs[0]
        
        # 6. Compute loss: negative log probability (to identify important tokens for vulnerability detection)
        loss = -torch.log(vuln_prob + 1e-8)
        
        # 7. Backward pass
        loss.backward()
        
        # 8. Extract gradient magnitude per token (L2 norm of embedding gradients)
        if embeddings_input.grad is not None:
            gradient_magnitude = torch.norm(embeddings_input.grad, dim=-1)[0].detach().cpu().numpy()
        else:
            # Fallback: uniform distribution if no gradients computed
            gradient_magnitude = np.ones(inputs['input_ids'].shape[1])
    
    # 9. Get tokens
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].detach())
    
    return tokens, gradient_magnitude


print("✅ Gradient-based Saliency 함수 정의 완료")

In [0]:
# ============================================================================
# Step 6: Integrated Gradients (가장 정확한 방법 - 추천)
# ============================================================================

def compute_integrated_gradients(
    code_text: str,
    model,
    tokenizer: AutoTokenizer,
    device: torch.device,
    n_steps: int = 20,
    baseline_type: str = 'zero'
) -> Tuple[List[str], np.ndarray]:
    """
    Integrated Gradients: 중요도의 수학적으로 가장 견고한 방법
    """
    model.eval()
    
    # Get base model and embedding layer (handle different model structures)
    if hasattr(model, 'base'):
        base_model = model.base
    elif hasattr(model, 'model'):
        base_model = model.model
    else:
        base_model = model
    
    # Try to get embedding layer using HuggingFace standard method
    try:
        embedding_layer = model.get_input_embeddings()
    except:
        # Fallback: try common attribute names
        if hasattr(base_model, 'embeddings'):
            embedding_layer = base_model.embeddings.word_embeddings if hasattr(base_model.embeddings, 'word_embeddings') else base_model.embeddings
        else:
            raise AttributeError("Cannot find embedding layer in model")
    
    # 1. Tokenize
    inputs = tokenizer(
        code_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=Config.MAX_LENGTH
    ).to(device)
    
    input_ids = inputs['input_ids']
    
    # 2. Get embeddings
    target_embeddings = embedding_layer(input_ids)
    
    # 3. Create baseline
    if baseline_type == 'zero':
        baseline = torch.zeros_like(target_embeddings)
    elif baseline_type == 'mask':
        mask_token_id = tokenizer.mask_token_id
        baseline_ids = torch.full_like(input_ids, mask_token_id)
        baseline = embedding_layer(baseline_ids)
    else:  # 'pad'
        pad_token_id = tokenizer.pad_token_id
        baseline_ids = torch.full_like(input_ids, pad_token_id)
        baseline = embedding_layer(baseline_ids)
    
    # 4. Compute accumulated gradients along the path
    accumulated_grads = torch.zeros_like(target_embeddings)
    
    for step in range(n_steps):
        # Interpolate between baseline and target
        alpha = step / n_steps
        interpolated_embeddings = baseline + alpha * (target_embeddings - baseline)
        # Enable gradients on interpolated embeddings (floating point)
        interpolated_embeddings = interpolated_embeddings.clone().detach().requires_grad_(True)
        interpolated_embeddings.retain_grad()
        
        # Forward pass through base model
        try:
            outputs = base_model(
                inputs_embeds=interpolated_embeddings,
                attention_mask=inputs['attention_mask'],
                token_type_ids=inputs.get('token_type_ids', None)
            )
        except TypeError:
            # Fallback if inputs_embeds not supported
            outputs = base_model(input_ids=input_ids, attention_mask=inputs['attention_mask'])
        
        # Get model outputs for vulnerability prediction
        if hasattr(outputs, 'logits'):
            logits = outputs.logits
        elif 'vul_logits' in outputs:
            logits = outputs['vul_logits']
        else:
            logits = outputs[0]
        
        # Handle different tensor shapes
        if logits.dim() == 1:
            logits = logits.unsqueeze(0)
        elif logits.dim() > 2:
            logits = logits.view(-1, logits.size(-1))
        
        # Get vulnerability probability
        vuln_probs = torch.softmax(logits, dim=-1)[0]
        vuln_prob = vuln_probs[1] if len(vuln_probs) > 1 else vuln_probs[0]
        
        # Compute loss: negative log probability
        loss = -torch.log(vuln_prob + 1e-8)
        
        # Compute gradients
        loss.backward()
        
        if interpolated_embeddings.grad is not None:
            accumulated_grads += interpolated_embeddings.grad.detach()
        
        # Zero out gradients for next iteration
        if interpolated_embeddings.grad is not None:
            interpolated_embeddings.grad.zero_()
    
    # 5. Compute integrated gradients
    integrated_grads = (target_embeddings - baseline) * accumulated_grads / n_steps
    
    # 6. Aggregate across embedding dimension (L2 norm)
    importance = torch.norm(integrated_grads[0], dim=-1).detach().cpu().numpy()
    
    # 7. Get tokens
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].detach())
    
    return tokens, importance


print("✅ Integrated Gradients 함수 정의 완료")

In [0]:
# ============================================================================
# Step 7: Attention Weight Visualization (가장 해석 가능한 방법)
# ============================================================================

def extract_attention_weights(
    code_text: str,
    model,
    tokenizer: AutoTokenizer,
    device: torch.device
) -> Tuple[List[str], np.ndarray]:
    """
    Extract and aggregate attention weights from all layers
    """
    model.eval()
    
    # Get base model (handle different model structures)
    if hasattr(model, 'base'):
        base_model = model.base
    elif hasattr(model, 'model'):
        base_model = model.model
    else:
        base_model = model
    
    # 1. Tokenize
    inputs = tokenizer(
        code_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=Config.MAX_LENGTH
    ).to(device)
    
    # 2. Forward pass with output_attentions=True
    with torch.no_grad():
        try:
            outputs = base_model(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                token_type_ids=inputs.get('token_type_ids', None),
                output_attentions=True
            )
        except TypeError:
            # Try without token_type_ids if not supported
            outputs = base_model(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                output_attentions=True
            )
    
    # 3. Extract attention weights
    attentions = outputs.attentions if hasattr(outputs, 'attentions') else outputs[-1]
    
    # Handle case where attentions might be nested or differently structured
    if not isinstance(attentions, tuple) and not isinstance(attentions, list):
        attentions = [attentions]
    
    # 4. Aggregate attention from [CLS] token across all layers and heads
    cls_attentions = []
    
    for layer_attn in attentions:
        try:
            # layer_attn shape: [batch_size, num_heads, seq_len, seq_len]
            if layer_attn.dim() == 4:
                cls_to_all = layer_attn[0, :, 0, :]  # [num_heads, seq_len]
                mean_attn = cls_to_all.mean(dim=0)   # [seq_len]
            else:
                # Handle unexpected shapes by flattening
                mean_attn = layer_attn[0].mean(dim=0) if layer_attn.dim() > 1 else layer_attn
            
            cls_attentions.append(mean_attn)
        except Exception as e:
            print(f"⚠️  Error processing layer attention: {e}")
            continue
    
    # 5. Aggregate across layers
    if len(cls_attentions) > 0:
        final_attention = torch.stack(cls_attentions).mean(dim=0).cpu().numpy()
    else:
        # Fallback: use uniform distribution
        seq_len = inputs['input_ids'].shape[1]
        final_attention = np.ones(seq_len) / seq_len
    
    # 6. Get tokens
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    return tokens, final_attention

print("✅ Attention Weight Visualization 함수 정의 완료")

## Step 7: Attention Weight Visualization (가장 해석 가능한 방법)


In [0]:
# Example vulnerable code (SQL Injection)
test_code = """
fn = function(context) {
  try {
    for(var i = 0, ii = length, part; i<ii; i++) {
      if (typeof (part = parts[i]) == 'function') {
        part = part(context);
        if (part == null || part == undefined) {
          part = '';
          } else if (typeof part != 'string') {
            part = toJson(part);
          }
        }
        concat[i] = part;
      }
    return concat.join('');
  }
  catch(err) {
    var newErr = minErr('$interpolate')('interr', \"Can't interpolate: {0}\\n{1}\", text, err.toString());$exceptionHandler(newErr);
  }
};
"""

print("📝 Input Code:")
print("="*80)
print(test_code)
print("="*80)

## Step 8: Perform Vulnerability Detection

In [0]:
print("\n" + "="*80)
print("🔍 Vulnerability Detection")
print("="*80 + "\n")

prediction_result = predict_vulnerability(test_code, model, tokenizer, device)

if prediction_result is not None:
    print(f"\n✅ Prediction Result:")
    print(f"   Status: {prediction_result['prediction']}")
    print(f"   Vulnerability Probability: {prediction_result['vulnerability_prob']:.4f}")
    print(f"   Confidence: {prediction_result['confidence']:.4f}")
else:
    print("❌ Prediction failed!")

# Debug: Model structure inspection
print("\n" + "="*80)
print("🔧 Model Structure Inspection (for XAI functions)")
print("="*80)
print(f"Model type: {type(model).__name__}")
print(f"Model attributes: {[attr for attr in dir(model) if not attr.startswith('_')][:15]}")

# Find embedding layer
embedding_layer = None
if hasattr(model, 'embeddings'):
    embedding_layer = model.embeddings
    print(f"✅ Found: model.embeddings")
elif hasattr(model, 'base') and hasattr(model.base, 'embeddings'):
    embedding_layer = model.base.embeddings
    print(f"✅ Found: model.base.embeddings")
elif hasattr(model, 'model') and hasattr(model.model, 'embeddings'):
    embedding_layer = model.model.embeddings
    print(f"✅ Found: model.model.embeddings")
elif hasattr(model, 'bert') and hasattr(model.bert, 'embeddings'):
    embedding_layer = model.bert.embeddings
    print(f"✅ Found: model.bert.embeddings")
else:
    print(f"⚠️  Embedding layer not found directly. Trying alternative approach...")
    print(f"   Model keys: {list(model.state_dict().keys())[:10]}")

print("="*80 + "\n")

## Step 9: XAI Analysis - 3가지 Input Sensitivity 방법 비교 실행


In [0]:
print("\n" + "="*80)
print("🔬 3가지 Input Sensitivity 방법 실행 (병렬 분석)")
print("="*80 + "\n")

# 1. Gradient-based Saliency
print("⏳ Method 1: Computing Gradient-based Saliency...")
tokens_sal, saliency_scores = compute_gradient_saliency(test_code, model, tokenizer, device)
print(f"✅ Gradient Saliency: {len(tokens_sal)} tokens 분석 완료")

# 2. Integrated Gradients
print("⏳ Method 2: Computing Integrated Gradients...")
tokens_ig, ig_scores = compute_integrated_gradients(test_code, model, tokenizer, device, n_steps=20)
print(f"✅ Integrated Gradients: {len(tokens_ig)} tokens 분석 완료")

# 3. Attention Weights
print("⏳ Method 3: Extracting Attention Weights...")
tokens_attn, attn_scores = extract_attention_weights(test_code, model, tokenizer, device)
print(f"✅ Attention Weights: {len(tokens_attn)} tokens 분석 완료")

print("\n✅ 3가지 방법 모두 분석 완료!\n")

# Normalize scores to [0, 1]
def normalize_scores(scores):
    min_val = scores.min()
    max_val = scores.max()
    if max_val == min_val:
        return np.ones_like(scores)
    return (scores - min_val) / (max_val - min_val)

saliency_norm = normalize_scores(saliency_scores)
ig_norm = normalize_scores(ig_scores)
attn_norm = normalize_scores(attn_scores)

print("📊 Score Normalization 완료")


In [0]:
code_to_analyze = """
fn = function(context) {
  try {
    for(var i = 0, ii = length, part; i<ii; i++) {
      if (typeof (part = parts[i]) == 'function') {
        part = part(context);
        if (part == null || part == undefined) {
          part = '';
          } else if (typeof part != 'string') {
            part = toJson(part);
          }
        }
        concat[i] = part;
      }
    return concat.join('');
  }
  catch(err) {
    var newErr = minErr('$interpolate')('interr', \"Can't interpolate: {0}\\n{1}\", text, err.toString());$exceptionHandler(newErr);
  }
};
"""

print("📝 Code to Analyze:")
print("="*80)
print(code_to_analyze)
print("="*80)

In [0]:
print("\n" + "="*80)
print("📊 XAI Visualization Results")
print("="*80 + "\n")

# Color function
def get_color_for_score(score_norm: float) -> str:
    """Return RGB color based on normalized importance score"""
    if score_norm > 0.4:
        return 'ff6b6b'  # Red - Critical
    elif score_norm > 0.25:
        return 'ffa94d'  # Orange - Medium
    else:
        return '51cf66'  # Green - Low

# Compute average importance from the 3 methods
# avg_importance = (saliency_norm + ig_norm + attn_norm) / 3
avg_importance = (ig_norm + attn_norm) / 2

# Create token-score mapping
token_score_map = {}
max_avg_score = max(avg_importance) if len(avg_importance) > 0 else 1

for i, token in enumerate(tokens_sal):
    # Skip special tokens
    if token.strip() not in ['[CLS]', '[SEP]', '<s>', '</s>', '[PAD]', '']:
        token_clean = token.replace('##', '').lower()
        if len(token_clean) > 1 and i < len(avg_importance):
            score_normalized = avg_importance[i] / max_avg_score if max_avg_score > 0 else 0
            # Keep max score for duplicate tokens
            if token_clean not in token_score_map or score_normalized > token_score_map[token_clean]:
                token_score_map[token_clean] = score_normalized

# Display original code
print("📝 ORIGINAL CODE:")
print("-" * 80)
print(code_to_analyze)
print()

# Highlight each line
highlighted_lines = []
for line in code_to_analyze.split('\n'):
    highlighted_line = html_module.escape(line)  # First escape HTML special chars
    
    # Replace each important token with colored span
    for token_key, norm_score in token_score_map.items():
        color = get_color_for_score(norm_score)
        
        # Word boundary matching to avoid partial matches
        pattern = r'\b' + re.escape(token_key) + r'\b'
        replacement = (
            f'<span style="background-color: #{color}; color: white; '
            f'font-weight: bold; padding: 2px 4px; border-radius: 2px;">'
            f'{html_module.escape(token_key)}</span>'
        )
        highlighted_line = re.sub(pattern, replacement, highlighted_line, flags=re.IGNORECASE)
    
    highlighted_lines.append(highlighted_line)

highlighted_code = '\n'.join(highlighted_lines)

# Display highlighted code
print("\n🎯 HIGHLIGHTED CODE (Important Tokens Marked):")
print("-" * 80)

html_output = f"""
<div style='background-color: #0d1117; color: #e0e0e0; padding: 15px; border-radius: 5px;
            font-family: "Courier New", monospace; line-height: 1.6; overflow-x: auto;
            border: 2px solid #2196F3;'>
<div style="margin-bottom: 10px;">
    <strong style="color: #58a6ff;">Color Guide:</strong>
    <span style="background-color: #51cf66; color: white; padding: 2px 4px; margin-left: 8px; border-radius: 2px;">🟢 Low Importance</span>
    <span style="background-color: #ffa94d; color: white; padding: 2px 4px; margin-left: 8px; border-radius: 2px;">🟠 Medium Importance</span>
    <span style="background-color: #ff6b6b; color: white; padding: 2px 4px; margin-left: 8px; border-radius: 2px;">🔴 Critical Importance</span>
</div>
<hr style="border: 1px solid #444; margin: 10px 0;">
<pre>{highlighted_code}</pre>
</div>
"""

display(HTML(html_output))

print("\n✅ Visualization Complete!")
print(f"   - Code length: {len(code_to_analyze)} characters")
print(f"   - Important tokens highlighted: {len(token_score_map)}")
print(f"   - Average importance score: {avg_importance.mean():.4f}")

## Step 11: 종합 비교 및 상관관계 분석


In [0]:
print("\n" + "="*80)
print("📊 종합 비교 분석: 3가지 방법간 상관관계")
print("="*80 + "\n")

from scipy.stats import spearmanr, pearsonr

# Correlation analysis
if len(saliency_scores) > 2:
    # Spearman rank correlation
    corr_sal_ig_s, p_sal_ig_s = spearmanr(saliency_scores, ig_scores)
    corr_sal_attn_s, p_sal_attn_s = spearmanr(saliency_scores, attn_scores)
    corr_ig_attn_s, p_ig_attn_s = spearmanr(ig_scores, attn_scores)
    
    # Pearson correlation
    corr_sal_ig_p, p_sal_ig_p = pearsonr(saliency_scores, ig_scores)
    corr_sal_attn_p, p_sal_attn_p = pearsonr(saliency_scores, attn_scores)
    corr_ig_attn_p, p_ig_attn_p = pearsonr(ig_scores, attn_scores)
    
    print("🔗 Spearman Rank Correlation (순서 일관성):")
    print(f"   Gradient Saliency ↔ Integrated Gradients: {corr_sal_ig_s:.4f} (p-value: {p_sal_ig_s:.4f})")
    print(f"   Gradient Saliency ↔ Attention Weights:   {corr_sal_attn_s:.4f} (p-value: {p_sal_attn_s:.4f})")
    print(f"   Integrated Gradients ↔ Attention Weights: {corr_ig_attn_s:.4f} (p-value: {p_ig_attn_s:.4f})")
    
    print("\n📈 Pearson Correlation (수치 일관성):")
    print(f"   Gradient Saliency ↔ Integrated Gradients: {corr_sal_ig_p:.4f} (p-value: {p_sal_ig_p:.4f})")
    print(f"   Gradient Saliency ↔ Attention Weights:   {corr_sal_attn_p:.4f} (p-value: {p_sal_attn_p:.4f})")
    print(f"   Integrated Gradients ↔ Attention Weights: {corr_ig_attn_p:.4f} (p-value: {p_ig_attn_p:.4f})")
    
    # Average correlation
    avg_corr = np.mean([corr_sal_ig_s, corr_sal_attn_s, corr_ig_attn_s])
    print(f"\n✨ 평균 상관계수 (Spearman): {avg_corr:.4f}")
    
    if avg_corr > 0.7:
        consistency = "⭐⭐⭐ 매우 높음 (신뢰도 우수)"
    elif avg_corr > 0.5:
        consistency = "⭐⭐ 중간 (신뢰도 양호)"
    else:
        consistency = "⭐ 낮음 (방법별 보완 필요)"
    
    print(f"   일관성 평가: {consistency}")

# Top token overlap analysis
print("\n\n🎯 Top-K 토큰 겹침 분석:")

top_k_check = min(5, Config.TOP_K_TOKENS)
set_sal = set(top_tokens_sal[:top_k_check])
set_ig = set(top_tokens_ig[:top_k_check])
set_attn = set(top_tokens_attn[:top_k_check])

overlap_sal_ig = len(set_sal & set_ig)
overlap_sal_attn = len(set_sal & set_attn)
overlap_ig_attn = len(set_ig & set_attn)
overlap_all = len(set_sal & set_ig & set_attn)

print(f"   Top-{top_k_check} 토큰 중 공통 토큰:")
print(f"   - Gradient Saliency ∩ Integrated Gradients: {overlap_sal_ig}/{top_k_check}")
print(f"   - Gradient Saliency ∩ Attention Weights:   {overlap_sal_attn}/{top_k_check}")
print(f"   - Integrated Gradients ∩ Attention Weights: {overlap_ig_attn}/{top_k_check}")
print(f"   - 3가지 모두 겹침: {overlap_all}/{top_k_check}")

print("\n" + "="*80)


## Step 12: 시각화 - 3가지 방법 비교 그래프


In [0]:
import matplotlib.pyplot as plt

print("\n" + "="*80)
print("📊 3가지 방법 시각화")
print("="*80 + "\n")

fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.3)

# Plot 1: Token importance comparison (line plot)
ax1 = fig.add_subplot(gs[0, :])
token_range = range(min(25, len(tokens_sal)))
x_labels = [tokens_sal[i][:12] for i in token_range]

ax1.plot(token_range, saliency_norm[:25], marker='o', label='Gradient Saliency', linewidth=2, markersize=5)
ax1.plot(token_range, ig_norm[:25], marker='s', label='Integrated Gradients', linewidth=2, markersize=5)
ax1.plot(token_range, attn_norm[:25], marker='^', label='Attention Weights', linewidth=2, markersize=5)

ax1.set_xlabel('Token Position', fontsize=11, fontweight='bold')
ax1.set_ylabel('Normalized Importance', fontsize=11, fontweight='bold')
ax1.set_title('Token Importance: 3가지 방법 비교 (앞 25개 토큰)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10, loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_xticks(token_range[::3])
ax1.set_xticklabels(x_labels[::3], rotation=45, ha='right')

# Plot 2: Top-10 comparison - Gradient Saliency
ax2 = fig.add_subplot(gs[1, 0])
top_indices_sal = np.argsort(saliency_scores)[-10:][::-1]
ax2.barh(range(10), saliency_scores[top_indices_sal], color='steelblue', alpha=0.8, edgecolor='black')
ax2.set_yticks(range(10))
ax2.set_yticklabels([tokens_sal[i][:15] for i in top_indices_sal], fontsize=9)
ax2.set_xlabel('Score', fontsize=10)
ax2.set_title('Top 10 - Gradient Saliency\n(⚡ 가장 빠름)', fontsize=11, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

# Plot 3: Top-10 comparison - Integrated Gradients
ax3 = fig.add_subplot(gs[1, 1])
top_indices_ig = np.argsort(ig_scores)[-10:][::-1]
ax3.barh(range(10), ig_scores[top_indices_ig], color='coral', alpha=0.8, edgecolor='black')
ax3.set_yticks(range(10))
ax3.set_yticklabels([tokens_ig[i][:15] for i in top_indices_ig], fontsize=9)
ax3.set_xlabel('Score', fontsize=10)
ax3.set_title('Top 10 - Integrated Gradients\n(🏆 가장 정확함)', fontsize=11, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')

# Plot 4: Top-10 comparison - Attention Weights
ax4 = fig.add_subplot(gs[2, 0])
top_indices_attn = np.argsort(attn_scores)[-10:][::-1]
ax4.barh(range(10), attn_scores[top_indices_attn], color='lightgreen', alpha=0.8, edgecolor='black')
ax4.set_yticks(range(10))
ax4.set_yticklabels([tokens_attn[i][:15] for i in top_indices_attn], fontsize=9)
ax4.set_xlabel('Score', fontsize=10)
ax4.set_title('Top 10 - Attention Weights\n(👁️ 가장 해석 가능)', fontsize=11, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='x')

# Plot 5: Distribution comparison
ax5 = fig.add_subplot(gs[2, 1])
ax5.hist(saliency_norm, bins=20, alpha=0.6, label='Gradient Saliency', color='steelblue', edgecolor='black')
ax5.hist(ig_norm, bins=20, alpha=0.6, label='Integrated Gradients', color='coral', edgecolor='black')
ax5.hist(attn_norm, bins=20, alpha=0.6, label='Attention Weights', color='lightgreen', edgecolor='black')
ax5.set_xlabel('Normalized Importance', fontsize=10)
ax5.set_ylabel('Token Count', fontsize=10)
ax5.set_title('점수 분포 비교', fontsize=11, fontweight='bold')
ax5.legend(fontsize=9)
ax5.grid(True, alpha=0.3, axis='y')

plt.suptitle('GraphCodeBERT XAI: 3가지 Input Sensitivity 분석 방법 비교', 
             fontsize=14, fontweight='bold', y=0.995)
plt.show()

print("✅ 시각화 완료!")


## Step 13: 결과 요약 및 권장 사항


In [0]:
print("\n" + "="*80)
print("📌 3가지 방법 결과 요약 및 선택 기준")
print("="*80 + "\n")

summary_table = """
┌─────────────────────┬──────────────┬──────────────┬─────────────┬──────────┐
│     방법            │    속도      │   정확도     │  해석성     │  추천도  │
├─────────────────────┼──────────────┼──────────────┼─────────────┼──────────┤
│ Gradient Saliency   │ ⚡ 매우 빠름 │ ⭐⭐⭐     │ ⭐⭐⭐    │ ⭐⭐⭐ │
│                     │ (1회 추론)   │              │             │  (균형)  │
├─────────────────────┼──────────────┼──────────────┼─────────────┼──────────┤
│ Integrated Gradients│ 🔄 중간     │ ⭐⭐⭐⭐⭐ │ ⭐⭐⭐    │ ⭐⭐⭐⭐⭐│
│                     │ (20회 추론)  │ (가장 정확)  │             │  (최추천) │
├─────────────────────┼──────────────┼──────────────┼─────────────┼──────────┤
│ Attention Weights   │ ⚡ 매우 빠름 │ ⭐⭐⭐     │ ⭐⭐⭐⭐⭐│ ⭐⭐⭐⭐│
│                     │ (이미 포함)  │              │ (최고 해석) │ (보조)   │
└─────────────────────┴──────────────┴──────────────┴─────────────┴──────────┘
"""

print(summary_table)

print("\n💡 사용 시나리오별 권장:")
print("─" * 80)
print("""
1️⃣  빠른 분석 필요 (실시간 API):
   → Attention Weights (1회 추론)
   
2️⃣  가장 정확한 분석 필요:
   → Integrated Gradients (권장)
   
3️⃣  균형잡힌 분석:
   → Gradient Saliency + Attention (병행)
   
4️⃣  최고 신뢰도 필요:
   → Integrated Gradients + Attention Weights (교차 검증)
""")

print("\n🎯 현재 분석 결과 요약:")
print("─" * 80)
print(f"   입력 코드 길이: {len(test_code)} 자")
print(f"   토크나이즈 토큰 수: {len(tokens_sal)}")
print(f"   분석된 토큰 수: {len(saliency_scores)} (Saliency), {len(ig_scores)} (IG), {len(attn_scores)} (Attention)")
print(f"   Top-1 중요 토큰:")
print(f"      • Gradient Saliency:    {top_tokens_sal[0]} (score: {top_scores_sal[0]:.6f})")
print(f"      • Integrated Gradients: {top_tokens_ig[0]} (score: {top_scores_ig[0]:.6f})")
print(f"      • Attention Weights:    {top_tokens_attn[0]} (score: {top_scores_attn[0]:.6f})")

print("\n" + "="*80)


---

## 📌 워크플로우 설명 및 방법론

### 🔄 실행 순서

1. **Step 1-4**: 모델 및 기본 함수 로드
2. **Step 5-7**: 3가지 XAI 분석 함수 정의
3. **Step 8**: 입력 코드 설정
4. **Step 9-10**: 기본 예측 및 분석 실행
5. **Step 9**: 3가지 방법 병렬 실행
6. **Step 10**: 방법별 Top-K 토큰 추출
7. **Step 11**: 방법간 상관관계 분석
8. **Step 12**: 시각화
9. **Step 13**: 결과 요약 및 권장사항
10. **Step 14**: 결과 내보내기 (선택)

### 🎯 3가지 XAI 방법론

#### 1️⃣ **Gradient-based Saliency Maps** (1순위 속도)
- **원리**: 손실함수에 대한 입력의 그래디언트 계산
- **수식**: `∂Loss/∂Input`
- **장점**: ⚡ 매우 빠름 (1회 추론), 직관적
- **단점**: Saturation 문제 가능
- **사용 시**: 실시간 분석 필요할 때

#### 2️⃣ **Integrated Gradients** (1순위 정확도 - 권장 🏆)
- **원리**: Baseline에서 Input까지의 경로 상 그래디언트 누적
- **수식**: `IG(x) = (x - x_baseline) × ∫₀¹ ∇f(x_baseline + t·(x - x_baseline)) dt`
- **장점**: 수학적으로 엄밀 (Axiom 만족), 가장 정확
- **단점**: N번 추론 필요 (보통 20회)
- **사용 시**: 높은 정확도 필요할 때 (권장)

#### 3️⃣ **Attention Weight Visualization** (1순위 해석성)
- **원리**: Transformer의 multi-head attention weights 직접 활용
- **수식**: `[CLS] 토큰 → 각 토큰으로의 attention 누적`
- **장점**: 🚀 가장 빠름 (추가 계산 없음), 모델의 "생각" 직접 확인
- **단점**: 토큰 간 상호작용만 반영
- **사용 시**: 해석 가능성 중시할 때

### 💡 방법 선택 가이드

```
빠른 분석?
  ├─ YES → Attention Weights or Gradient Saliency
  └─ NO → Integrated Gradients

신뢰도 중심?
  ├─ 최고: Integrated Gradients
  ├─ 높음: Integrated Gradients + Attention
  └─ 중간: Gradient Saliency

해석성 중심?
  ├─ 최고: Attention Weights
  ├─ 높음: Attention + Gradient Saliency
  └─ 중간: Integrated Gradients
```

### 📊 각 방법의 특징

| 항목 | Gradient Saliency | Integrated Gradients | Attention Weights |
|------|------------------|----------------------|-------------------|
| **수행 시간** | <1초 | ~5초 | <1초 |
| **정확도** | 중간 | 높음 ⭐⭐⭐⭐⭐ | 중간 |
| **해석성** | 높음 | 중간 | 매우 높음 ⭐⭐⭐⭐⭐ |
| **계산복잡도** | O(1) | O(N) | O(1) |
| **신뢰도** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ |

### 🔍 결과 해석

**높은 점수 토큰**:
- 모델의 취약점 예측에 **크게 영향**
- 빨간색(🔴)으로 표시되면 **매우 중요**
- 이 토큰들이 코드의 취약성 지표

**낮은 점수 토큰**:
- 상대적으로 **덜 중요**
- 초록색(🟢)으로 표시
- 배경 정보로 간주 가능

### 📈 추천 분석 조합

**최적 조합**: Integrated Gradients (메인) + Attention (보조)
- 정확도 + 해석성 최적화
- 상호 검증으로 신뢰도 향상

**추천 코드**:
```python
# Step 9 이후에 다음 코드 추가 가능
# 두 방법의 공통 중요 토큰 찾기
common_tokens = set(top_tokens_ig[:5]) & set(top_tokens_attn[:5])
print(f"두 방법 모두에서 중요: {common_tokens}")
```
